<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/12-caching-memory/01-unified-memory-model.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 12.1 — Unified memory model: watch storage lose to execution

Cache a DataFrame so it fills most of available memory, then run a
shuffle-heavy stage alongside it and watch Storage Memory get evicted
via the Executors tab REST API — the unified pool in action, not just
in a diagram.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark import StorageLevel

spark = (
    SparkSession.builder
    .appName("12.1-unified-memory-model")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")  # stable teaching plans (Module 7/10 convention)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
# local[*] note: one JVM plays driver + all executors, so the Executors tab
# shows a single "driver" entry — the memory MODEL below is identical to a
# real cluster, only the topology is collapsed.

## The Executors tab as data: `memoryUsed` vs `maxMemory`

This is the *current* boundary between storage's claim and execution's
claim inside the unified pool — it moves during a run, it's not a
static config value.

In [ ]:
import urllib.request, json as _json

def executor_memory_report():
    app_id = spark.sparkContext.applicationId
    url = f"http://localhost:4040/api/v1/applications/{app_id}/executors"
    for e in _json.load(urllib.request.urlopen(url)):
        used, mx = e["memoryUsed"], e["maxMemory"]
        print(f"executor {e['id']}: storage memory used {used:,} / {mx:,} "
              f"({100*used/mx:.1f}%)")

executor_memory_report()   # baseline — nothing cached yet

## Fill storage memory with a cache

Cache something big enough to actually show up as a meaningful
fraction of `maxMemory`.

In [ ]:
big = spark.range(20_000_000).withColumn("k", (col("id") % 50).cast("int")) \
                              .withColumn("pad", F.lit("x" * 200))
big.persist(StorageLevel.MEMORY_ONLY)
big.count()
print("after caching:")
executor_memory_report()

## Now run a shuffle-heavy stage alongside it

A sort over a large range needs execution-memory scratch space. Watch
whether storage's reported usage changes — that's eviction pressure,
not a bug.

In [ ]:
shuffle_heavy = spark.range(20_000_000).withColumn("v", (col("id") * 2654435761) % 1000000007)
shuffle_heavy.orderBy("v").count()   # forces a big sort — competes for execution memory

print("after the shuffle-heavy sort:")
executor_memory_report()
# compare to the post-cache snapshot above — did storage give ground?

## `spark.memory.storageFraction` — the protected floor

Lower the floor and repeat: with less of the pool protected, storage
should be squeezed harder by the same shuffle.

In [ ]:
big.unpersist()
spark.conf.set("spark.memory.storageFraction", "0.1")   # was 0.5 — shrink the protected floor

big.persist(StorageLevel.MEMORY_ONLY)
big.count()
print("cached under storageFraction=0.1:")
executor_memory_report()

shuffle_heavy.orderBy("v").count()
print("\nafter the same shuffle-heavy sort, smaller floor:")
executor_memory_report()

spark.conf.set("spark.memory.storageFraction", "0.5")   # restore default
big.unpersist()

## The separate Python-worker budget (9.0's callback)

`spark.executor.pyspark.memory` is not part of the JVM heap this
lesson diagrammed — print it to see it's a distinct, separately
configured pool.

In [ ]:
for k in ["spark.executor.memory", "spark.memory.fraction",
          "spark.memory.storageFraction", "spark.executor.pyspark.memory"]:
    try:
        print(k, "=", spark.conf.get(k))
    except Exception:
        print(k, "= <unset, uses JVM/runtime default>")

In [ ]:
spark.stop()

## Exercises

1. Repeat the cache-then-shuffle experiment with `storageFraction` set
   very high (`0.9`). Does storage survive the shuffle better? What
   would you expect to happen to the shuffle's spill (12.2) as a
   trade-off?
2. Cache two different DataFrames, one right after another, each
   larger than available memory. Use `storage_report()` (12.0) and
   `executor_memory_report()` together — which one gets evicted first,
   and does that match LRU?
3. `spark.memory.fraction` controls how much of the heap is "Spark
   memory" at all (vs user memory). Lower it to `0.3` and repeat the
   baseline cache. Does `maxMemory` in the report change? Why or why
   not (what does `maxMemory` actually measure)?
4. Design an experiment that shows execution memory borrowing *idle*
   storage space and giving it back — i.e., a case where storage
   is NOT evicted because the shuffle didn't actually need the room.
5. Explain, from what you observed, why a job's cache can shrink
   mid-run with literally zero new code executing.

In [ ]:
# your exercise code here